# Talking to the MCP Server Directly

No LLM and no agent: just an MCP client that
1. lists the tools the server offers, and
2. calls the `health` tool.

```
┌──────────────────────────┐   list_tools()     ┌──────────────────────┐
│ MCP client (notebook)    ├───────────────────►│ MCP server           │
│                          │   call_tool()      │ (Databricks App)     │
│                          │◄───────────────────┤                      │
└──────────────────────────┘   results          └──────────────────────┘
```

In [0]:
%pip install -q "databricks-mcp" "databricks-sdk>=0.50.0"
dbutils.library.restartPython()

## Configuration and authentication

Same service principal setup (OAuth M2M) as in the other notebooks.

In [0]:
PROFILE = "resa-prod"                # Databricks CLI profile (only used with LOCAL_MCP)
APP_NAME = "mcp-server-template"    # Databricks App hosting the MCP server
LOCAL_MCP = False                   # True -> use `make run` server on localhost:8000

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

if LOCAL_MCP:
    w = WorkspaceClient(profile=PROFILE)
    mcp_url = "http://localhost:8000/mcp"
else:
    w = WorkspaceClient(
        host=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get(),  # current workspace
        client_id=dbutils.secrets.get("service-principals", "mcp-server-template-user-client-id"),
        client_secret=dbutils.secrets.get("service-principals", "mcp-server-template-user-client-secret"),
        auth_type="oauth-m2m",
    )
    mcp_url = w.apps.get(APP_NAME).url.rstrip("/") + "/mcp"

# Handles OAuth for the request using the WorkspaceClient's credentials
mcp = DatabricksMCPClient(server_url=mcp_url, workspace_client=w)

print("MCP URL:", mcp_url)

## 1. List the tools

In [0]:
# Needs async version because mcp.list_tools() calls asyncio.run() internally and Notebook runs its own event loop
tools = await mcp.alist_tools()

for tool in tools:
    print(f"- {tool.name}: {(tool.description or '').strip().splitlines()[0]}")

## 2. Call the `health` tool

In [0]:
# Needs async version because mcp.call_tool() calls asyncio.run() internally and Notebook runs its own event loop
result = await mcp.acall_tool("health", {})

print(result.content[0].text)